In [1]:
import pandas as pd
import numpy as np
import os

# Load raw data exactly as-is, all columns as string first to avoid silent type coercion
raw_path = "../data/raw/Medical-Equipment-Suppliers.csv"
df = pd.read_csv(raw_path, dtype=str, low_memory=False)

print("File:", raw_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print()
print("Column names:")
print(df.columns.tolist())

File: ../data/raw/Medical-Equipment-Suppliers.csv
Rows: 57197
Columns: 17

Column names:
['provider_id', 'acceptsassignement', 'participationbegindate', 'businessname', 'practicename', 'practiceaddress1', 'practiceaddress2', 'practicecity', 'practicestate', 'practicezip9code', 'telephonenumber', 'specialitieslist', 'providertypelist', 'supplieslist', 'latitude', 'longitude', 'is_contracted_for_cba']


In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report.sort_values("missing_count", ascending=False)
print(missing_report)

                        missing_count  missing_pct
providertypelist                50445        88.20
practiceaddress2                44376        77.58
specialitieslist                  752         1.31
supplieslist                       26         0.05
provider_id                         0         0.00
practicezip9code                    0         0.00
longitude                           0         0.00
latitude                            0         0.00
telephonenumber                     0         0.00
practicestate                       0         0.00
acceptsassignement                  0         0.00
practicecity                        0         0.00
practiceaddress1                    0         0.00
practicename                        0         0.00
businessname                        0         0.00
participationbegindate              0         0.00
is_contracted_for_cba               0         0.00


In [2]:
# Data types as originally read (all string) and memory usage
print(df.dtypes)
print()
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1_000_000, 2))

provider_id               object
acceptsassignement        object
participationbegindate    object
businessname              object
practicename              object
practiceaddress1          object
practiceaddress2          object
practicecity              object
practicestate             object
practicezip9code          object
telephonenumber           object
specialitieslist          object
providertypelist          object
supplieslist              object
latitude                  object
longitude                 object
is_contracted_for_cba     object
dtype: object

Memory usage (MB): 74.3


In [4]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate provider_id values:", df["provider_id"].duplicated().sum())
print("Distinct provider_id count:", df["provider_id"].nunique())

Exact duplicate rows: 0
Duplicate provider_id values: 0
Distinct provider_id count: 57197


In [5]:
print(df["acceptsassignement"].value_counts(dropna=False))
print()
print(df["acceptsassignement"].value_counts(normalize=True, dropna=False).round(4) * 100)

acceptsassignement
True     29416
False    27781
Name: count, dtype: int64

acceptsassignement
True     51.43
False    48.57
Name: proportion, dtype: float64


In [6]:
print("Distinct states:", df["practicestate"].nunique())
print(df["practicestate"].value_counts().head(10))
print()
dates = pd.to_datetime(df["participationbegindate"], errors="coerce")
print("Earliest date:", dates.min())
print("Latest date:", dates.max())
print("Rows before year 2000:", (dates.dt.year < 2000).sum())
print("Unparseable dates:", dates.isna().sum())

Distinct states: 55
practicestate
NY    4883
CA    4419
TX    4382
FL    4020
PA    2161
IL    2098
OH    2013
NC    1955
GA    1834
NJ    1781
Name: count, dtype: int64

Earliest date: 1940-01-02 00:00:00
Latest date: 2026-07-01 00:00:00
Rows before year 2000: 6915
Unparseable dates: 0


In [7]:
for col in ["practicestate", "practicecity", "specialitieslist", "supplieslist"]:
    has_whitespace = df[col].dropna().apply(lambda x: x != x.strip()).sum()
    print(f"{col}: {has_whitespace} values with leading/trailing whitespace")

practicestate: 0 values with leading/trailing whitespace
practicecity: 0 values with leading/trailing whitespace
specialitieslist: 0 values with leading/trailing whitespace
supplieslist: 0 values with leading/trailing whitespace


In [8]:
os.makedirs("../reports/validation", exist_ok=True)

summary_rows = [
    {"check": "total_rows", "value": len(df)},
    {"check": "total_columns", "value": len(df.columns)},
    {"check": "distinct_provider_id", "value": df["provider_id"].nunique()},
    {"check": "duplicate_provider_id_rows", "value": int(df["provider_id"].duplicated().sum())},
    {"check": "exact_duplicate_rows", "value": int(df.duplicated().sum())},
    {"check": "target_true_count", "value": int((df["acceptsassignement"] == "True").sum())},
    {"check": "target_false_count", "value": int((df["acceptsassignement"] == "False").sum())},
    {"check": "distinct_states", "value": df["practicestate"].nunique()},
    {"check": "earliest_participation_date", "value": str(dates.min())},
    {"check": "latest_participation_date", "value": str(dates.max())},
    {"check": "rows_before_year_2000", "value": int((dates.dt.year < 2000).sum())},
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("../reports/validation/data_quality_report.csv", index=False)

missing_report.to_csv("../reports/validation/missing_values_report.csv")

print("Exported:")
print("- reports/validation/data_quality_report.csv")
print("- reports/validation/missing_values_report.csv")
summary_df

Exported:
- reports/validation/data_quality_report.csv
- reports/validation/missing_values_report.csv


,check,value
0,total_rows,57197
1,total_columns,17
2,distinct_provider_id,57197
3,duplicate_provider_id_rows,0
4,exact_duplicate_rows,0
5,target_true_count,29416
6,target_false_count,27781
7,distinct_states,55
8,earliest_participation_date,1940-01-02 00:00:00
9,latest_participation_date,2026-07-01 00:00:00


In [9]:
for col in ["practicestate", "practicecity", "specialitieslist", "supplieslist"]:
    has_whitespace = df[col].dropna().apply(lambda x: x != x.strip()).sum()
    print(f"{col}: {has_whitespace} values with leading/trailing whitespace")

practicestate: 0 values with leading/trailing whitespace
practicecity: 0 values with leading/trailing whitespace
specialitieslist: 0 values with leading/trailing whitespace
supplieslist: 0 values with leading/trailing whitespace


In [ ]:
s